# Tools and web search for agents
An agent is a model that can also **act**. In this notebook the agent gets tools: first small Python functions (square root, square), then a real **web search** so it can answer things it could not know from training.

Only `GOOGLE_API_KEY` and `TAVILY_API_KEY` are needed (from `.env`, Colab Secrets or pasted below). No OpenAI key is used. The example questions are about Algeria.

**Without tools:** the model answers only from what it learned in training.
```
Request  -->  Model  -->  Result
```

**With tools:** the model can decide to use a tool, look at what came back, and decide again, until it has an answer.
```
Request
   |
   v
+----------------------- agent -----------------------+
|   Model ---- Action (calls a tool) ----> Tools      |
|   Model <--- Observation (tool result) -- Tools      |
+----------------------------|-------------------------+
                             v
                          Result
```
- **Action** = the model asks for a tool call (name + arguments).
- **Observation** = what the tool returned, sent back to the model as a `ToolMessage`.

## Setup

In [ ]:
!pip install -q langchain langchain-google-genai tavily-python python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\ProgramData\anaconda3\python.exe -m pip install --upgrade pip


In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()  # local run: reads the .env file (returns False if none is found)

KEYS = ("GOOGLE_API_KEY", "TAVILY_API_KEY")

# Colab run: read the keys from the Secrets panel (key icon in the left sidebar)
try:
    from google.colab import userdata
    for key in KEYS:
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
except ImportError:
    pass

def is_real(value):
    # empty values and template placeholders ("your_..._here") count as missing
    return bool(value) and not value.startswith("your_")

# Still missing? Paste it here (hidden input).
for key in KEYS:
    if not is_real(os.getenv(key)):
        try:
            value = getpass(f"{key} is missing - paste it (Enter to skip): ").strip()
        except Exception:
            value = ""
        if value:
            os.environ[key] = value

for key in KEYS:
    print(f"{key}: {'set' if is_real(os.getenv(key)) else 'MISSING'}")

# Tracing with a placeholder LangSmith key only produces 401/403 warnings: switch it off
if not is_real(os.getenv("LANGSMITH_API_KEY")):
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing: off (no valid LANGSMITH_API_KEY)")

GOOGLE_API_KEY: set
TAVILY_API_KEY: set
LangSmith tracing: off (no valid LANGSMITH_API_KEY)


In [ ]:
from datetime import date
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.tools import tool
from pprint import pprint

MODEL = "google_genai:gemini-3.1-flash-lite"

def show(title, text):
    print(f"\n{'=' * 12} {title} {'=' * 12}\n{text}")

def show_messages(response):
    """Print EVERY message of the run, not just the final answer."""
    for i, m in enumerate(response["messages"]):
        print(f"[{i}] {type(m).__name__}")
        print("    content   :", repr(m.text))
        if getattr(m, "tool_calls", None):
            print("    tool_calls:", m.tool_calls)

def tools_used(response):
    """Names of the tools the agent called during the run."""
    return [tc["name"] for m in response["messages"] for tc in getattr(m, "tool_calls", None) or []]

def run(agent, text):
    """Ask one question, print the tools called and the final answer."""
    response = agent.invoke({"messages": [HumanMessage(text)]})
    show("Question", text)
    show("Tools called", tools_used(response) or "none")
    show("Answer", response["messages"][-1].text)
    return response

## Part 1: Tools
A **tool** is a normal Python function that the agent is allowed to call. The model never sees the code inside: it only sees the **name**, the **description** (the docstring) and the **type hints** of the arguments.

> **`@tool`**: turns a function into a tool. Name = function name, description = docstring.

In [ ]:
@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

Two other ways to write the same tool: give the name explicitly, or give the name **and** the description (then no docstring is needed).

In [ ]:
@tool("square_root")
def sqrt_way2(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool("square_root", description="Calculate the square root of a number")
def sqrt_way3(x: float) -> float:
    return x ** 0.5

What the model actually sees, and a direct test of the tool (Algeria covers 2,381,741 km2):

In [ ]:
print("name       :", square_root.name)
print("description:", square_root.description)
print("arguments  :", square_root.args)

square_root.invoke({"x": 2381741})

name       : square_root
description: Calculate the square root of a number
arguments  : {'x': {'title': 'X', 'type': 'number'}}


1543.289020242158

### Adding tools to an agent
`create_agent(model=..., tools=[...])`. The system prompt tells the agent when to use them.

In [ ]:
agent = create_agent(
    model=MODEL,
    tools=[square_root, square],
    system_prompt="You are an arithmetic wizard. Use your tools to calculate the square root and square of any number.",
)

question = HumanMessage(content="Algeria covers 2,381,741 km2. What is the square root of that number?")

response = agent.invoke({"messages": [question]})

print(response["messages"][-1].text)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


The square root of Algeria's land area of 2,381,741 km² is approximately 1,543.29.


### Task 1: build a tool and watch the messages
Print **every** message, not only the final answer. There are four messages:
1. `HumanMessage`: the question.
2. `AIMessage` with **empty content** and a `tool_calls` list: the *Action*. The model does not answer yet, it asks for a tool.
3. `ToolMessage`: the tool result, the *Observation*.
4. `AIMessage`: the final answer, written after seeing the observation.

> **`tool_calls`**: list on an `AIMessage` with the tool `name`, the `args` and an `id`.  **`ToolMessage`**: the result of a tool call, sent back to the model.

In [ ]:
show_messages(response)

[0] HumanMessage
    content   : 'Algeria covers 2,381,741 km2. What is the square root of that number?'
[1] AIMessage
    content   : ''
    tool_calls: [{'name': 'square_root', 'args': {'x': 2381741}, 'id': 'call_55555', 'type': 'tool_call'}]
[2] ToolMessage
    content   : '1543.289020242158'
[3] AIMessage
    content   : "The square root of Algeria's land area of 2,381,741 km² is approximately 1,543.29."


In [ ]:
# The tool call inside the AI message
print(response["messages"][1].tool_calls)

[{'name': 'square_root', 'args': {'x': 2381741}, 'id': 'call_55555', 'type': 'tool_call'}]


In [ ]:
pprint(response['messages'])

[HumanMessage(content='Algeria covers 2,381,741 km2. What is the square root of that number?', additional_kwargs={}, response_metadata={}, id='30c78a18-4cb5-443b-a341-9df8b2fe001b'),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'square_root', 'arguments': '{"x": 2381741}'}, '__gemini_function_call_thought_signatures__': {'call_55555': 'EnEKbwFpFH0TF3Bfx2hfQtLAwo//fR0ATx0TaU30EdukDsK283JSo3znSYRy07ZJskLiCdIPwSFXP+p/C162/z8Tgg5Ri4f6KamB4Pz74CvHh6fs7k5+u6pUCp/8ia1wSFS22CLtDFDfgNYLTsL4z4GFDw=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0d012-26f1-7730-8ca9-375651a56891-0', tool_calls=[{'name': 'square_root', 'args': {'x': 2381741}, 'id': 'call_55555', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 131, 'output_tokens': 20, 'total_tokens': 151, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='1543.289020242

### Task 2: write the description badly on purpose
The agent chooses a tool **only from its name and description**. Here two tools have vague names and the one-word description `math`. The agent cannot tell which one does what.

In [ ]:
@tool("do_it", description="math")
def bad_tool_1(x: float) -> float:
    return x ** 0.5      # really a square root

@tool("do_it_2", description="math")
def bad_tool_2(x: float) -> float:
    return x ** 2        # really a square

agent_bad = create_agent(model=MODEL, tools=[bad_tool_1, bad_tool_2])

q_root = "What is the square root of 2381741?"
response_bad = run(agent_bad, q_root)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.



============ Question ============
What is the square root of 2381741?

============ Tools called ============
['do_it_2', 'do_it']

============ Answer ============
The square root of 2,381,741 is approximately 1,543.289.


Check the result: the square root of 2,381,741 is about **1543.29**. If the agent used `do_it_2`, the number is huge (a square). Run the cell a few times: with vague descriptions the choice is a guess. Now the same two tools with a good name and description:

In [ ]:
@tool("square_root", description="Calculate the square root of a number. Use it when the user asks for a square root.")
def good_tool_1(x: float) -> float:
    return x ** 0.5

@tool("square", description="Calculate the square of a number (x times x). Use it when the user asks for a square.")
def good_tool_2(x: float) -> float:
    return x ** 2

agent_good = create_agent(model=MODEL, tools=[good_tool_1, good_tool_2])

response_good = run(agent_good, q_root)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.



============ Question ============
What is the square root of 2381741?

============ Tools called ============
['square_root']

============ Answer ============
The square root of 2381741 is approximately 1543.289.


**What I learned:** the description is the interface between the model and my code. A clear name and a one-sentence description that says *what it does* and *when to use it* is the skill.

## Part 2: Web search
### Task 3: find your model's edge
A model only knows what it saw in training. Ask something it cannot know (this week's news, a recent fact) and watch how it answers. Then give it a search tool and ask again.

**Step 1: without web search**

In [ ]:
agent_no_search = create_agent(model=MODEL)

questions = [
    "How up to date is your training knowledge?",
    "Who is the current president of Algeria?",
    "What are the latest news headlines in Algeria this week?",
]

for q in questions:
    run(agent_no_search, q)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.



============ Question ============
How up to date is your training knowledge?

============ Tools called ============
none

============ Answer ============
My knowledge cutoff is **January 2025**. Therefore, I cannot provide you with any information about events or developments that have occurred since that time.

============ Question ============
Who is the current president of Algeria?

============ Tools called ============
none

============ Answer ============
The current president of Algeria is **Abdelmadjid Tebboune**. He has been serving in the position since December 2019 and was re-elected for a second term in September 2024.

============ Question ============
What are the latest news headlines in Algeria this week?

============ Tools called ============
none

============ Answer ============
As of mid-May 2024, the news cycle in Algeria is dominated by preparations for the upcoming presidential election, international diplomatic efforts, and socio-economic developments.

Read the answers carefully: does it admit that it cannot know, or does it answer with confidence? A confident answer that is out of date is the real danger.

**Step 2: add a web search tool** (Tavily). The `@tool` function just calls the search service and returns the result.

> **`TavilyClient().search(query)`**: sends a query to the Tavily search API and returns a dict with a list of `results` (title, url, content).

In [ ]:
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()  # reads TAVILY_API_KEY from the environment

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for up-to-date information: news, current office holders, prices and recent events."""
    return tavily_client.search(query)

results = web_search.invoke("Who is the current president of Algeria?")

for r in results["results"][:3]:
    print("-", r["title"], "\n ", r["url"])

- President of Algeria | Current Leader 
  https://planetrulers.com/algeria-president
- Abdelmadjid Tebboune - Wikipedia 
  https://en.wikipedia.org/wiki/Abdelmadjid_Tebboune
- Algerian President Abdelmadjid Tebboune strengthens his grip on power 
  https://www.lemonde.fr/en/international/article/2026/03/25/algerian-president-abdelmadjid-tebboune-strengthens-his-grip-on-power_6751808_4.html


**Step 3: give the tool to the agent** and ask the same questions again. The system prompt gives today's date, because the model does not know it (needed for "this week").

In [ ]:
agent_search = create_agent(
    model=MODEL,
    tools=[web_search],
    system_prompt=f"You are a helpful assistant. Today's date is {date.today():%B %d, %Y}. Use web_search for anything recent or that may have changed.",
)

for q in questions[1:]:
    run(agent_search, q)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.



============ Question ============
Who is the current president of Algeria?

============ Tools called ============
['web_search']

============ Answer ============
The current president of Algeria is **Abdelmadjid Tebboune**. He has been serving in this position since December 2019.

============ Question ============
What are the latest news headlines in Algeria this week?

============ Tools called ============
['web_search']

============ Answer ============
As of September 23, 2026, the latest news headlines in Algeria focus on government policy, international diplomacy, and economic updates:

*   **Judicial & Security Policy:** Authorities have confirmed the enforcement of the death penalty for drug traffickers, signaling a toughened stance on narcotics-related crimes. 
*   **Diplomatic Activity:** 
    *   Algerian Foreign Minister Ahmed Attaf has been in New York, participating in a consultative meeting of Arab foreign ministers. 
    *   The Ministry of Foreign Affairs is act

Watch the messages of one run: the model calls `web_search`, the tool returns a long observation, then the model writes the answer from it.

In [ ]:
response = agent_search.invoke({"messages": [HumanMessage("Who is the current president of Algeria?")]})

show_messages(response)

[0] HumanMessage
    content   : 'Who is the current president of Algeria?'
[1] AIMessage
    content   : ''
    tool_calls: [{'name': 'web_search', 'args': {'query': 'current president of Algeria'}, 'id': 'call_67324', 'type': 'tool_call'}]
[2] ToolMessage
    content   : '{"query": "current president of Algeria", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://planetrulers.com/algeria-president", "title": "President of Algeria | Current Leader - Planet Rulers", "content": "Abdelmadjid Tebboune is the eighth President of the People’s Democratic Republic of Algeria, in office since 19 December 2019. He was re-elected to a second five-year term on 7 September 2024 with an officially reported 84.3% of the vote (later amended upward by the Constitutional Council). Tebboune also holds the portfolio of Minister of Defence, making him the country’s civilian-military apex alongside the Army Chief of Staff. Long-time career civil servant — wali (governor

## What I learned
1. **A tool is a function plus a description.** The model sees the name, the description and the argument types, never the code.
2. **The agent works in a loop:** the model asks for a tool (*Action*), the tool answers (*Observation*), and the model decides again until it can give the *Result*.
3. **Watch the messages.** The AI message that calls a tool has empty content and a `tool_calls` list. Printing every message is the best way to debug an agent.
4. **The description is the skill.** With vague names and a one-word description the agent has to guess which tool to use.
5. **A model has an edge.** Without search it may answer recent questions with confidence and be wrong, or admit it does not know. A search tool gives it access to current information.
6. **Put today's date in the system prompt** when the question depends on "now".

**Mistakes to avoid:** never share API keys (revoke and recreate any key that leaks); run the cells from the top; in Colab the `.env` file on your PC is not visible, use Secrets or paste the key.

## Summary of functions
| Function / class | Simple definition |
|---|---|
| `@tool` | Turns a Python function into a tool an agent can call. |
| `@tool("name", description="...")` | Same, with an explicit name and description. |
| `tool.name`, `.description`, `.args` | What the model sees about the tool. |
| `tool.invoke({...})` | Runs the tool directly, without a model. |
| `create_agent(model, tools=[...])` | Builds an agent that can call the listed tools. |
| `response["messages"]` | All messages of the run: human, AI (with tool calls), tool results, final AI. |
| `message.tool_calls` | The tools the model asked to run: name, args, id. |
| `ToolMessage` | The result of a tool call, sent back to the model (the observation). |
| `TavilyClient().search(query)` | Web search API: returns titles, links and content. |
| `show_messages()`, `tools_used()`, `run()`, `show()` | Helpers written in this notebook to print messages, list tools used and ask a question. |

By
> Kassouar Fatima              24-09-2026:05:30



